#Project 1: Premier League Table Recreation

This notebook analyizes the premier League Table from the 2025/2026 season

In [1]:
import pandas as pd

df_PlData = pd.read_csv('PLProject1Data.csv')

In [2]:
df_PlData.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,B365CAHH,B365CAHA,PCAHH,PCAHA,MaxCAHH,MaxCAHA,AvgCAHH,AvgCAHA,BFECAHH,BFECAHA
0,E0,15/08/2025,20:00,Liverpool,Bournemouth,4,2,H,1,0,...,2.03,1.78,2.07,1.85,2.03,1.88,1.94,1.76,2.14,1.86
1,E0,16/08/2025,12:30,Aston Villa,Newcastle,0,0,D,0,0,...,2.05,1.80,2.02,1.89,2.06,1.80,1.95,1.74,2.14,1.86
2,E0,16/08/2025,15:00,Brighton,Fulham,1,1,D,0,0,...,1.83,2.03,1.93,2.00,1.84,2.03,1.80,1.96,1.91,2.08
3,E0,16/08/2025,15:00,Sunderland,West Ham,3,0,H,0,0,...,1.95,1.90,1.97,1.95,1.95,1.94,1.86,1.78,2.02,1.97
4,E0,16/08/2025,15:00,Tottenham,Burnley,3,0,H,1,0,...,1.98,1.88,1.99,1.93,1.98,1.91,1.88,1.83,2.07,1.92


Goals scored by Team

In [3]:

# Goals scored at home
home_goals = df_PlData.groupby("HomeTeam")["FTHG"].sum()

# Goals scored away
away_goals = df_PlData.groupby("AwayTeam")["FTAG"].sum()

# Combine both
total_goals = home_goals.add(away_goals, fill_value=0)

print(total_goals.sort_values(ascending=False))


HomeTeam
Man City          77
Arsenal           71
Man United        69
Liverpool         63
Bournemouth       58
Chelsea           58
Aston Villa       56
Brentford         55
Newcastle         53
Brighton          52
Leeds             49
Nott'm Forest     48
Tottenham         48
Fulham            47
Everton           47
West Ham          46
Sunderland        42
Crystal Palace    41
Burnley           38
Wolves            27
dtype: int64


Which team conceded the fewest

In [4]:
# Goals conceded at home (away team scores)
home_conceded = df_PlData.groupby("HomeTeam")["FTAG"].sum()

# Goals conceded away (home team scores)
away_conceded = df_PlData.groupby("AwayTeam")["FTHG"].sum()

# Total conceded
total_conceded = home_conceded.add(away_conceded, fill_value=0)

print(total_conceded.sort_values(ascending=True))


HomeTeam
Arsenal           27
Man City          35
Brighton          46
Sunderland        48
Aston Villa       49
Everton           50
Man United        50
Fulham            51
Nott'm Forest     51
Crystal Palace    51
Brentford         52
Chelsea           52
Liverpool         53
Bournemouth       54
Newcastle         55
Leeds             56
Tottenham         57
West Ham          65
Wolves            68
Burnley           75
dtype: int64


Home vs away performance

In [5]:
# Filter only matches where the home team won
home_wins = df_PlData[df_PlData["FTHG"] > df_PlData["FTAG"]]

# Count wins per home team
home_win_counts = home_wins.groupby("HomeTeam").size()

print(home_win_counts.sort_values(ascending=False))


HomeTeam
Arsenal           15
Man City          14
Man United        13
Aston Villa       12
Fulham            11
Liverpool         10
Newcastle         10
Sunderland         9
Brighton           9
Leeds              9
Brentford          8
Bournemouth        7
Chelsea            7
West Ham           6
Everton            6
Crystal Palace     4
Nott'm Forest      4
Tottenham          3
Wolves             3
Burnley            2
dtype: int64


In [6]:
# Filter only matches where the away team won
away_wins = df_PlData[df_PlData["FTAG"] > df_PlData["FTHG"]]

# Count wins per away team
away_win_counts = away_wins.groupby("AwayTeam").size()

print(away_win_counts.sort_values(ascending=False))



AwayTeam
Arsenal           11
Man City           9
Aston Villa        7
Nott'm Forest      7
Chelsea            7
Everton            7
Crystal Palace     7
Liverpool          7
Man United         7
Tottenham          7
Bournemouth        6
Brentford          6
Sunderland         5
Brighton           5
Fulham             4
Newcastle          4
West Ham           4
Leeds              2
Burnley            2
dtype: int64


In [7]:
import pandas as pd

# --- GOALS SCORED ---
goals_scored = (
    df_PlData.groupby("HomeTeam")["FTHG"].sum()
    .add(df_PlData.groupby("AwayTeam")["FTAG"].sum(), fill_value=0)
)

# --- GOALS CONCEDED ---
goals_conceded = (
    df_PlData.groupby("HomeTeam")["FTAG"].sum()
    .add(df_PlData.groupby("AwayTeam")["FTHG"].sum(), fill_value=0)
)

# --- HOME WINS ---
home_wins = df_PlData[df_PlData["FTHG"] > df_PlData["FTAG"]].groupby("HomeTeam").size()

# --- AWAY WINS ---
away_wins = df_PlData[df_PlData["FTAG"] > df_PlData["FTHG"]].groupby("AwayTeam").size()

# Total wins
total_wins = home_wins.add(away_wins, fill_value=0)

# --- HOME LOSSES ---
home_losses = df_PlData[df_PlData["FTHG"] < df_PlData["FTAG"]].groupby("HomeTeam").size()

# --- AWAY LOSSES ---
away_losses = df_PlData[df_PlData["FTAG"] < df_PlData["FTHG"]].groupby("AwayTeam").size()

# Total losses
total_losses = home_losses.add(away_losses, fill_value=0)

# --- DRAWS ---
draws = df_PlData[df_PlData["FTHG"] == df_PlData["FTAG"]]
home_draws = draws.groupby("HomeTeam").size()
away_draws = draws.groupby("AwayTeam").size()
total_draws = home_draws.add(away_draws, fill_value=0)

# --- POINTS ---
points = total_wins * 3 + total_draws * 1

# --- BUILD FINAL TABLE ---
league_table = pd.DataFrame({
    "Team": goals_scored.index,
    "Wins": total_wins,
    "Losses": total_losses,
    "GoalsScored": goals_scored,
    "GoalsConceded": goals_conceded,
    "Points": points
})

# Replace NaN with 0 for teams missing any category
league_table = league_table.fillna(0)

# Sort by Points, then Goal Difference, then Goals Scored
league_table["GoalDifference"] = league_table["GoalsScored"] - league_table["GoalsConceded"]
league_table = league_table.sort_values(
    by=["Points", "GoalDifference", "GoalsScored"],
    ascending=False
)

# Add Position column
league_table.insert(0, "Position", range(1, len(league_table) + 1))

print(league_table)


                Position            Team  Wins  Losses  GoalsScored  \
HomeTeam                                                              
Arsenal                1         Arsenal  26.0       5           71   
Man City               2        Man City  23.0       6           77   
Man United             3      Man United  20.0       7           69   
Aston Villa            4     Aston Villa  19.0      11           56   
Liverpool              5       Liverpool  17.0      12           63   
Bournemouth            6     Bournemouth  13.0       7           58   
Sunderland             7      Sunderland  14.0      12           42   
Brighton               8        Brighton  14.0      13           52   
Brentford              9       Brentford  14.0      13           55   
Chelsea               10         Chelsea  14.0      14           58   
Fulham                11          Fulham  15.0      16           47   
Newcastle             12       Newcastle  14.0      17           53   
Everto